[< Back to Main README](../README.md) | [Demo README](./README.md)

# Hooks vs Agent Control — Block vs Self-Correct

## The Problem with Blocking

Hooks (functions that intercept agent actions) enforce business rules at the tool level — when a rule fails, `cancel_tool` blocks the call and the agent reports failure. The user then has to rephrase or adjust their request manually.

But the agent could have fixed the problem itself. If the user asks for 15 guests and the max is 10, the agent could adjust to 10 and complete the booking — instead of just saying "I can't do that."

## The Solution: Steer Instead of Block

[Agent Control](https://github.com/agentcontrol/agent-control) introduces **steer controls** — when a violation is detected, the agent receives corrective guidance via `Guide()` and retries with the fix applied:

| | Hooks | Agent Control |
|---|---|---|
| Where rules live | Python code (`hooks=[...]`) | Server — API/dashboard |
| When rule fails | `cancel_tool = "BLOCKED"` → agent fails | `Guide("reduce to 10")` → agent retries corrected |
| To change a rule | Edit code, redeploy | API call — no code changes |
| Integration | `HookProvider` + `hooks=[...]` | `Plugin` + `plugins=[...]` |

## The Tools

Three booking tools in `tools.py` — clean, no validation logic:

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `book_hotel(hotel, check_in, check_out, guests)` | Books a hotel room | Returns `"SUCCESS: Booking BK001..."` — no guest limit in the tool |
| `process_payment(amount, booking_id)` | Processes payment | Returns `"SUCCESS"` or `"ERROR: Booking not found"` |
| `confirm_booking(booking_id)` | Confirms a booking | Returns `"SUCCESS: Confirmed BK001"` |

The tools do NOT enforce the max-guests rule. That is the guardrail layer's job.

## What We Test

Same query, same tools, same model — only the guardrail changes:

| Test | Guardrail | Expected behavior |
|------|-----------|-------------------|
| 1 — Hooks | `MaxGuestsHook` with `cancel_tool` | Agent is BLOCKED, asks user what to do |
| 2 — Agent Control | `AgentControlSteeringHandler` with `Guide()` | Agent self-corrects to 10 guests, completes booking |

This notebook compares two guardrail approaches for AI agents: **Hooks** (block invalid operations) vs **Agent Control** (steer the agent to self-correct). Both approaches are tested on the same scenario — booking a hotel with 15 guests when the maximum is 10.

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any provider supported by Strands — see [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) for configuration.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    'OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

In [ ]:
import os, time
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.hooks import HookProvider, HookRegistry, BeforeToolCallEvent

from tools import book_hotel, process_payment, confirm_booking, ALL_TOOLS

load_dotenv()

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# Same query for both tests
QUERY = "Book Grand Hotel for 15 guests from 2026-05-01 to 2026-05-03"

# Prompt that makes the LLM describe the booking before calling the tool
PROMPT = (
    "You are a hotel booking assistant. "
    "When booking, first describe what you will book (hotel, guests, dates) "
    "then call the tool."
)

print("Setup complete!")

---
## Test 1 — Hooks (Block with cancel_tool)

`MaxGuestsHook` intercepts `BeforeToolCallEvent` and checks the `guests` parameter. If > 10, it sets `event.cancel_tool` — the agent receives this as the tool result and must report the failure to the user.

**Query:** *"Book Grand Hotel for 15 guests from 2026-05-01 to 2026-05-03"*

> The agent will be blocked and ask the user what to do — the booking will NOT complete.

In [ ]:
class MaxGuestsHook(HookProvider):
    def __init__(self):
        self.blocked = 0
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)
    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "book_hotel":
            return
        guests = event.tool_use["input"].get("guests", 1)
        if guests > 10:
            self.blocked += 1
            event.cancel_tool = f"BLOCKED: {guests} guests exceeds maximum of 10"

hook = MaxGuestsHook()
agent_hooks = Agent(model=MODEL, system_prompt=PROMPT, tools=ALL_TOOLS, hooks=[hook])

start = time.time()
response_hooks = agent_hooks(QUERY)
time_hooks = time.time() - start

print(f"Time: {time_hooks:.1f}s")
print(f"Hook blocked: {hook.blocked} call(s)")
print(f"Outcome: {'BLOCKED' if hook.blocked > 0 else 'completed'}")

---
## Test 2 — Agent Control (Steer with Guide)

Agent Control evaluates the LLM output before tool execution. When the model says "I will book for 15 guests", the steer control detects the number > 10 and returns `Guide("reduce to 10")`. The agent retries with corrected parameters.

**Same query as Test 1** — the only difference is the guardrail layer.

> Requires Agent Control server running. See [setup instructions](https://github.com/agentcontrol/agent-control).

### How Steer Works

1. LLM generates: *"I will book Grand Hotel for 15 guests..."*
2. `AgentControlSteeringHandler` evaluates LLM output against server controls
3. Regex matches "15 guest" → steer control fires
4. Steering handler returns `Guide("reduce to 10 and inform user")`
5. LLM retries with guidance → calls `book_hotel(guests=10)`
6. Booking completes — user is informed about the adjustment

In [ ]:
try:
    import agent_control
    from agent_control.integrations.strands import AgentControlPlugin, AgentControlSteeringHandler
    from agent_control.control_decorators import ControlViolationError
    from strands.hooks import AfterToolCallEvent

    # Default server URL for local development; set AGENT_CONTROL_URL for production
    agent_control.init(
        agent_name="booking-guardrails-demo",
        server_url=os.getenv("AGENT_CONTROL_URL", "http://127.0.0.1:8000"),
    )

    plugin = AgentControlPlugin(
        agent_name="booking-guardrails-demo",
        event_control_list=[BeforeToolCallEvent, AfterToolCallEvent],
        enable_logging=False,
    )
    steering = AgentControlSteeringHandler(
        agent_name="booking-guardrails-demo",
        enable_logging=False,
    )

    agent_ac = Agent(model=MODEL, system_prompt=PROMPT, tools=ALL_TOOLS, plugins=[plugin, steering])

    start = time.time()
    response_ac = agent_ac(QUERY)
    time_ac = time.time() - start
    output_ac = str(response_ac)
    steered = steering.steers_applied

    print(f"Time: {time_ac:.1f}s")
    print(f"Steered: {steered} time(s)")
    print(f"Outcome: {'self-corrected' if 'BK' in output_ac else 'check output'}")

except ImportError as e:
    print(f"Missing: {e}")
    print("Run: uv pip install 'agent-control-sdk[strands-agents]'")
    time_ac, steered, output_ac = 0, 0, "skipped"
except Exception as e:
    print(f"Error: {type(e).__name__}: {str(e)[:200]}")
    print("Make sure Agent Control server is running.")
    print("See: https://github.com/agentcontrol/agent-control")
    time_ac, steered, output_ac = 0, 0, "error"

In [ ]:
print(f"{'Approach':<35} {'Time':>8} {'Outcome':>20}")
print("-" * 65)

hooks_outcome = "blocked" if hook.blocked > 0 else "completed"
ac_outcome = "self-corrected" if isinstance(output_ac, str) and "BK" in output_ac else str(output_ac)

print(f"{'Test 1 - Hooks (cancel_tool)':<35} {time_hooks:>6.1f}s {hooks_outcome:>20}")
print(f"{'Test 2 - Agent Control (steer)':<35} {time_ac:>6.1f}s {ac_outcome:>20}")

print()
print("Key difference:")
print("  Hooks:         agent receives 'BLOCKED' -> asks user what to do")
print("  Agent Control: agent receives Guide('reduce to 10') -> retries -> booking completes")

---
## Summary

### When to Use Each Approach

| Approach | Best for |
|----------|----------|
| **Hooks** | Rules that MUST hard-block — no workaround allowed (e.g., payment before confirmation) |
| **Agent Control (steer)** | Rules where the agent CAN self-correct — adjust parameters, redact PII, fix formatting |
| **Agent Control (deny)** | Same as hooks but managed on a server — change rules without redeploying code |

### Why This Integration Is Simple

Both approaches integrate with a single line change:

```python
# Hooks:
agent = Agent(tools=[...], hooks=[MaxGuestsHook()])

# Agent Control:
agent = Agent(tools=[...], plugins=[AgentControlPlugin(...), AgentControlSteeringHandler(...)])
```

### References

#### Framework Documentation
- [Hooks](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/) — `BeforeToolCallEvent`, `cancel_tool`
- [Steering](https://strandsagents.com/docs/user-guide/concepts/plugins/steering/) — `Guide`, `Proceed`, `SteeringHandler`
- [Agent Control Plugin](https://strandsagents.com/docs/community/plugins/agent-control/) — Integration docs
- [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) — Swap to Amazon Bedrock, Anthropic, Ollama

#### Guardrail Systems
- [Agent Control GitHub](https://github.com/agentcontrol/agent-control) — Open source, Apache 2.0
- [Agent Control Docs](https://docs.agentcontrol.dev/) — Server setup and API reference

#### Code
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)